# Phase 26 — Mixed Classifiers (διαφορετικά μοντέλα για hazard/product)

Δοκιμάζουμε διαφορετικούς συνδυασμούς μοντέλων για hazard και product.

**Λογική:** Το hazard (10 κλάσεις) και product (22 κλάσεις) είναι διαφορετικής δυσκολίας.
Ίσως διαφορετικά μοντέλα να ταιριάζουν καλύτερα σε καθένα.

**Διαθέσιμα .npy files:**
- `bertbase_focal_test_hazard_probs.npy` / `bertbase_focal_test_product_probs.npy`
- `multitask_test_hazard_probs.npy` / `multitask_test_product_probs.npy`
- SVM predictions (παράγονται εδώ)

**Best so far:** 0.81728 (BERT-base Focal 0.5 + Multi-task 0.5)

In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')



In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

# Φόρτωση BERT-base Focal probs
bert_hazard_probs  = np.load('bertbase_focal_test_hazard_probs.npy')
bert_product_probs = np.load('bertbase_focal_test_product_probs.npy')

# Φόρτωση Multi-task probs
multi_hazard_probs  = np.load('multitask_test_hazard_probs.npy')
multi_product_probs = np.load('multitask_test_product_probs.npy')

# Ensemble (0.5/0.5) probs
ensemble_hazard_probs  = 0.5 * bert_hazard_probs  + 0.5 * multi_hazard_probs
ensemble_product_probs = 0.5 * bert_product_probs + 0.5 * multi_product_probs

print(f'Φορτώθηκαν probs!')
print(f'BERT-base hazard:  {bert_hazard_probs.shape}')
print(f'Multi-task hazard: {multi_hazard_probs.shape}')

In [ ]:
# SVM predictions (εκπαίδευση σε train+valid)
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

train_full['combined'] = (train_full['title'].apply(preprocess_text) + ' ' +
                          train_full['text'].str[:550].apply(preprocess_text))
test['combined'] = (test['title'].apply(preprocess_text) + ' ' +
                    test['text'].str[:550].apply(preprocess_text))

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2),
                         sublinear_tf=True, stop_words='english')
X_train_svm = tfidf.fit_transform(train_full['combined'])
X_test_svm  = tfidf.transform(test['combined'])

clf_svm_hazard = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_hazard.fit(X_train_svm, train_full['hazard-category'])

clf_svm_product = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_product.fit(X_train_svm, train_full['product-category'])

svm_test_hazard  = clf_svm_hazard.predict(X_test_svm)
svm_test_product = clf_svm_product.predict(X_test_svm)

print('SVM εκπαιδεύτηκε')

In [ ]:
def make_submission(hazard_preds, product_preds, filename):
    pd.DataFrame({
        'id': test['id'],
        'hazard-category': hazard_preds,
        'product-category': product_preds
    }).to_csv(filename, index=False)
    print(f' {filename}')

print('=== ΔΗΜΙΟΥΡΓΙΑ SUBMISSIONS ===\n')

# Συνδυασμός Α: Hazard=BERT-base Focal | Product=Multi-task
make_submission(
    le_hazard.inverse_transform(bert_hazard_probs.argmax(axis=1)),
    le_product.inverse_transform(multi_product_probs.argmax(axis=1)),
    'submission_hazard_bert_product_multi.csv'
)

# Συνδυασμός Β: Hazard=Multi-task | Product=BERT-base Focal
make_submission(
    le_hazard.inverse_transform(multi_hazard_probs.argmax(axis=1)),
    le_product.inverse_transform(bert_product_probs.argmax(axis=1)),
    'submission_hazard_multi_product_bert.csv'
)

# Συνδυασμός Γ: Hazard=SVM | Product=Ensemble (0.5/0.5)
make_submission(
    svm_test_hazard,
    le_product.inverse_transform(ensemble_product_probs.argmax(axis=1)),
    'submission_hazard_svm_product_ensemble.csv'
)

# Συνδυασμός Δ: Hazard=Ensemble | Product=SVM
make_submission(
    le_hazard.inverse_transform(ensemble_hazard_probs.argmax(axis=1)),
    svm_test_product,
    'submission_hazard_ensemble_product_svm.csv'
)

# Συνδυασμός Ε: Hazard=BERT-base Focal | Product=Ensemble (0.5/0.5)
make_submission(
    le_hazard.inverse_transform(bert_hazard_probs.argmax(axis=1)),
    le_product.inverse_transform(ensemble_product_probs.argmax(axis=1)),
    'submission_hazard_bert_product_ensemble.csv'
)

# Συνδυασμός ΣΤ: Hazard=Ensemble (0.5/0.5) | Product=BERT-base Focal
make_submission(
    le_hazard.inverse_transform(ensemble_hazard_probs.argmax(axis=1)),
    le_product.inverse_transform(bert_product_probs.argmax(axis=1)),
    'submission_hazard_ensemble_product_bert.csv'
)

print('\n=== ΣΥΝΟΨΗ ===')
print('submission_hazard_bert_product_multi.csv     → Hazard=BERT, Product=Multi-task')
print('submission_hazard_multi_product_bert.csv     → Hazard=Multi-task, Product=BERT')
print('submission_hazard_svm_product_ensemble.csv   → Hazard=SVM, Product=Ensemble')
print('submission_hazard_ensemble_product_svm.csv   → Hazard=Ensemble, Product=SVM')
print('submission_hazard_bert_product_ensemble.csv  → Hazard=BERT, Product=Ensemble')
print('submission_hazard_ensemble_product_bert.csv  → Hazard=Ensemble, Product=BERT')
print('\nBest so far: 0.81728 (Ensemble 0.5/0.5 για hazard ΚΑΙ product)')